<h1>Create initial images to be used in feather tutorial</h1>

In [ ]:
# import pkg_resources, os
# casa_data_dir = pkg_resources.resource_filename("casadata", "__data__")
# rc_file = open(os.path.expanduser("~/.casarc"), "a+")  # append mode
# rc_file.write("\nmeasures.directory: " + casa_data_dir)
# rc_file.close()

In [ ]:
# download base images
from toolviper.utils.data import download

download("feather_sim_sd_c1_pI.im")
download("feather_sim_vla_c1_pI.im")

In [ ]:
# from graphviper.dask.client import local_client
# viper_client = local_client(cores=4, memory_limit="4GB")

# import dask
# dask.config.set(scheduler="synchronous")
# dask.config.set(scheduler="threads")

<h2>Inputs to be specified by user</h2>

In [ ]:
# ra, dec size, should not exceed 4096 x 4096
imsize = [1024, 1024]

# number of channels
nchan = 16

# currently, there is only one polarization and it is I

In [ ]:
from xradio.image.image import make_empty_sky_image
import numpy as np

rad_per_arcsec = np.pi / 180 / 3600
skel_xds = make_empty_sky_image(
    phase_center=[0.6, -0.2],
    image_size=imsize,
    cell_size=[15 * rad_per_arcsec, 15 * rad_per_arcsec],
    frequency_coords=np.linspace(1.4e9, 1.5e9, nchan),
    pol_coords=["I"],
    time_coords=[0],
)
# does not have dim beam_param
skel_xds

In [ ]:
from xradio.image import open_image

sel_dict = {}
if imsize[0] < 4096:
    blc = 2048 - imsize[0] // 2
    l_slice = slice(blc, blc + imsize[0])
    sel_dict["l"] = l_slice
if imsize[1] < 4096:
    blc = 2048 - imsize[1] // 2
    m_slice = slice(blc, blc + imsize[1])
    sel_dict["m"] = m_slice
xds_sd_temp = open_image("feather_sim_sd_c1_pI.im").isel(sel_dict)
print("xds_sd_temp sky shape", xds_sd_temp.SKY.shape)
print("xds_sd_temp beam shape", xds_sd_temp.BEAM_FIT_PARAMS_SKY.shape)
print("xds_sd_temp beam values", xds_sd_temp.BEAM_FIT_PARAMS_SKY.values)
xds_sd_temp

In [ ]:
xds_int_temp = open_image("feather_sim_vla_c1_pI.im").isel(sel_dict)
# xds_int_temp
print("xds_int_temp sky shape", xds_int_temp.SKY.shape)
print("xds_int_temp beam shape", xds_int_temp.BEAM_FIT_PARAMS_SKY.shape)
print("xds_int_temp beam values", xds_int_temp.BEAM_FIT_PARAMS_SKY.values)
xds_int_temp

In [ ]:
import dask.array as da
import xarray as xr

dm = skel_xds.sizes
sky_da_zeros = da.zeros(
    [dm["time"], dm["frequency"], dm["polarization"], dm["l"], dm["m"]],
    dtype=np.float32,
)
sky_dims = list(skel_xds.dims)
sky_dims.remove("beam_params_label")
coords = ["time", "frequency", "polarization", "l", "m"]
sky_coords = {}
for c in coords:
    sky_coords[c] = skel_xds[c]
sky_xa_zeros = xr.DataArray(data=sky_da_zeros, coords=sky_coords, dims=sky_dims)
sky_xa_zeros

In [ ]:
beam_da_zeros = da.zeros(
    [dm["time"], dm["frequency"], dm["polarization"], dm["beam_params_label"]],
    dtype=np.float32,
)
beam_dims = ["time", "frequency", "polarization", "beam_params_label"]
beam_xa_zeros = xr.DataArray(
    beam_da_zeros.copy(),
    dims=beam_dims,
    coords={k: v for k, v in skel_xds.coords.items() if k in beam_dims + ["velocity"]},
)

beam_xa_zeros

In [ ]:
import copy

for i in (0, 1):
    xds = copy.deepcopy(skel_xds)
    xds["SKY"] = sky_xa_zeros.copy()
    xds["BEAM_FIT_PARAMS_SKY"] = beam_xa_zeros.copy()
    for j in range(0, nchan, 16):
        min_chan = j
        max_chan = min(j + 16, nchan)
        fx = xds_sd_temp if i == 0 else xds_int_temp
        xds["SKY"][{"frequency": slice(min_chan, max_chan)}] = fx["SKY"].values
        xds["SKY"].attrs = {"units": "Jy/beam"}
        print(f"xds {id(xds)}")
        xds["BEAM_FIT_PARAMS_SKY"][{"frequency": slice(min_chan, max_chan)}] = fx["BEAM_FIT_PARAMS_SKY"].values
        xds["BEAM_FIT_PARAMS_SKY"].attrs = {"units": "rad"}
    if i == 0:
        xds_sd = xds
    else:
        xds_int = xds
xds_int.BEAM_FIT_PARAMS_SKY.values

In [ ]:
bytes_in_dtype = {"float32": 4, "double": 8, "complex": 16}

# chunking_dims_sizes = {'frequency':int_xds["sky"].sizes['frequency']}
# memory_singleton_chunk = 3*np.product(np.array(list(chunking_dims_sizes.values())))
xds_sd["SKY"].sizes["frequency"]

singleton_chunk_sizes = dict(xds_sd["SKY"].sizes)
del singleton_chunk_sizes["frequency"]  # Remove dimensions that will be chuncked on.
fudge_factor = 1.1
n_images_in_memory = 3.0
memory_singleton_chunk = (
    n_images_in_memory
    * np.prod(np.array(list(singleton_chunk_sizes.values())))
    * fudge_factor
    * bytes_in_dtype[str(xds_sd["SKY"].dtype)]
    / (1024**3)
)


memory_singleton_chunk

In [ ]:
xds_sd["SKY"].sel(polarization="I").isel(frequency=0, time=0).plot()

In [ ]:
# This is a point source, so may not be obvious
# in this plot. It's a small dot near the center.
xds_int["SKY"].sel(polarization="I").isel(frequency=0, time=0).plot()

In [ ]:
# These are the input images for the next step

import os, shutil

from xradio.image import write_image

for xds, outfile in zip([xds_sd, xds_int], ["sd.zarr", "int.zarr"]):
    if os.path.exists(outfile):
        shutil.rmtree(outfile)
    write_image(xds, outfile, "zarr")
    print(f"Wrote {outfile}")
# xds_int
xds_sd

In [ ]:
# You now have the required input images. Run feather_tutorial_v2.ipynb to use them
# to make the final feather image.